# 005 · Volatility model (not started)

**The question.** How much will this stock's price swing over the next three months?

**The answer.** `vol_63_future`: the annualised volatility of daily returns over the next 63 trading days. Model its logarithm (risk_params v5). Leave out the 86 rows whose answer is 0 (no logarithm).

**The baseline to beat.** `vol_63`: the same measure over the past 63 days, assumed to continue.

**Measures.** mean absolute error and root mean squared error of the log, rank correlation, and the share of futures inside the 80% range (should be 80%).

**Status.** Empty on purpose: headings and instructions only. Fill it in feature 005
([docs/plan.md](../docs/plan.md), issue in the roadmap).

### Rules for every model notebook

- **Spec first.** Before filling this notebook, run `/speckit-specify` for this feature with
  `SPECIFY_FEATURE_DIRECTORY=specs/005-volatility-model` on branch `005-volatility-model` (README, "Working
  together"). The notebook is for trying ideas; the chosen model goes into
  `src/sentinel/models/` and runs by command, because `sentinel update` scores every company each
  night without a notebook (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/volatility.parquet` through `sentinel.eda.load("volatility")`, which keeps
  the development rows (dated 2022 or earlier). Learn and test only on rows with
  `eligible = true` and an answer. Never read `data/exports/reference/` (answers for 2023 on) and
  never compute anything for 2023 or later: that is the sealed holdout, tested once by feature 010
  (constitution IV).
- **Never use as features:** `vol_63_future`, `window_end`, `cik`, `as_of`, and the flags (`eligible`, `excluded`, `active_filer`, `latest`, `month_end`, `release_eve`).
- **Evaluation is fixed** (decision D60, `config/risk_params.yaml` v5, never edited): yearly
  walk-forward test years 2019, 2020, 2021, 2022; for each, train only on rows whose answer window
  ended before 1 January of that year (`sentinel.eda.purge_table` counts them); four levels:
  naive baseline → rule → classic model → LightGBM; choose the **simplest level within one
  standard error of the best**; report every test year, not only the average.
- **Show forecasts as a level and an 80% range, with the range scaled on all earlier years (V2-10)** with their hit rates on the test years (D62), never as a bare
  number.
- **Reproducible:** sort rows by `cik`, `as_of` before modelling; fixed seeds (`SEED = 0`); log
  each run in MLflow with the git commit and the export date. Two runs must give identical
  predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms). Never write to `data/`.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/005-volatility-model.ipynb`

### What the EDA found for this model (notebooks/004-eda-volatility.ipynb, Findings)

- The naive baseline: mean absolute log error 0.35, 0.49, 0.29, 0.25 in 2019–2022 (rank
  correlation 0.71, 0.40, 0.82, 0.86). 2020 (the crash) is the hard year.
- **Rule level ready-made:** log(future) = a + b × log(past), fitted on earlier years (b ≈ 0.78:
  volatility moves partly back to normal). It beat the baseline in 2019–2021.
- Strongest features: past volatility in its forms, drawdown and distance from the high, price
  level, loss-making, size, interest coverage, R&D intensity, cash share. For the *change* from
  past to future: market volatility (`spy_vol_63`), VIX and 2-year rate changes, unemployment.
- Redundant pairs: keep one of `close`/`log_close`, `addv20`/`log_addv20`,
  `market_cap`/`log_market_cap`, `vol_252`/`idio_vol_252`, `vol_21`/`range_vol_21`, insider
  counts/values.
- Price-only stocks are barely in training (3,385 rows): open question V2-26.

Data fixes proposed but **not yet applied** (docs/decisions.md, V2-22 to V2-24: foreign filers'
market value, impossible share counts, insider dollar typos): until the researcher decides and the
pipeline is fixed, cut extreme values at the 1st and 99th percentiles of the training rows and
note it.

## 1. Setup and data

*To do:* Load the development rows, keep eligible rows with an answer, list the features to use (the EDA's advice below), and print rows and companies per test year.

## 2. Walk-forward folds

*To do:* Build the four folds (test years 2019–2022) with the purge; print train and test rows per fold.

## 3. Level 0: the naive baseline

*To do:* Score the baseline on each test year with the chosen measures.

## 4. Level 1: a rule

*To do:* A simple, explainable rule (see the EDA's suggestion below); fit only on each fold's training rows.

## 5. Level 2: a classic model

*To do:* A linear or logistic model: one feature of each redundant pair, ratios cut at their 1st and 99th percentiles (on training rows), missing values filled with the training median plus a 'was missing' flag, standardised.

## 6. Level 3: LightGBM

*To do:* Gradient-boosted trees with fixed seeds and settings chosen by time-ordered validation inside the training years only (never on the test year).

## 7. Comparison and choice

*To do:* One table: each level × each test year, the mean and its standard error; pick the simplest level within one standard error of the best.

## 8. Calibration and bands

*To do:* Scale the 80% range on all earlier years' errors (V2-10); check that about 80% of futures fall inside it in each test year; five bands by predicted volatility with each band's median outcome (D47).

## 9. What the model uses

*To do:* Feature importance (LightGBM) or coefficients (classic); check that the strongest features make sense and none is a leak.

## 10. Findings

*To do:* In plain words: the chosen level and why, its skill per test year against the baseline, the bands and their hit rates, what to move into `src/sentinel/models/`, and open questions.